[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HisameOgasahara/deep-learning-diagnostics-and-improvement/blob/main/practice/18_3d_geometry_and_rendering.ipynb)

# 18. 3D geometry and rendering — projection, NeRF, and 3D Gaussian splatting

이전 3DGS section은 3D covariance를 만든 뒤 별도의 임의 alpha를 compositing했기 때문에, **3D Gaussian이 camera projection을 통해 screen-space ellipse가 되고 pixel opacity를 만드는 핵심 단계**가 빠져 있었다.

이번 버전은 rigid transform → perspective projection → NeRF volume rendering → 3D covariance → projection Jacobian → 2D covariance → pixel Gaussian weight → depth-ordered alpha compositing까지 연결한다.


In [ ]:
import math

import torch
import torch.nn.functional as F

torch.manual_seed(7)
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("device:", device)


## 1. World point → camera point → pixel

3D rendering의 첫 단계는 world coordinate를 camera coordinate로 바꾸고 camera intrinsic으로 pixel coordinate에 projection하는 것이다.


In [ ]:
points_world = torch.tensor(
    [
        [0.2, 0.1, 2.0],
        [-0.3, 0.2, 3.0],
    ],
    device=device,
)

R_camera = torch.eye(3, device=device)
t_camera = torch.tensor(
    [0.1, 0.0, 0.0],
    device=device,
)

points_camera = points_world @ R_camera.T + t_camera

fx = 100.0
fy = 100.0
cx = 32.0
cy = 32.0

pixel_u = fx * points_camera[:, 0] / points_camera[:, 2] + cx
pixel_v = fy * points_camera[:, 1] / points_camera[:, 2] + cy
pixels = torch.stack([pixel_u, pixel_v], dim=-1)

print("camera points:\n", points_camera)
print("pixels:\n", pixels)


## 2. NeRF ray samples and volume rendering

NeRF는 camera ray를 따라 여러 3D sample을 만들고 network가 각 sample의 density `sigma_i`와 color `c_i`를 출력한다. density를 alpha로 바꾼 뒤 앞 sample이 뒤 sample을 가리는 transmittance를 누적한다.


In [ ]:
ray_origin = torch.tensor([0.0, 0.0, 0.0], device=device)
ray_direction = F.normalize(
    torch.tensor([0.2, 0.1, 1.0], device=device),
    dim=0,
)
depths = torch.linspace(0.5, 3.0, 6, device=device)
ray_samples = ray_origin + depths[:, None] * ray_direction

density = torch.tensor(
    [0.2, 0.5, 1.0, 0.3, 0.1, 0.05],
    device=device,
)
color = torch.tensor(
    [
        [1.0, 0.0, 0.0],
        [0.8, 0.2, 0.0],
        [0.0, 0.0, 1.0],
        [0.2, 0.8, 0.2],
        [1.0, 1.0, 1.0],
        [0.5, 0.5, 0.5],
    ],
    device=device,
)

delta = torch.diff(depths)
delta = torch.cat([delta, delta[-1:]], dim=0)

alpha = 1 - torch.exp(-density * delta)
survival = torch.cat(
    [torch.ones(1, device=device), 1 - alpha + 1e-8]
)
transmittance = torch.cumprod(survival, dim=0)[:-1]
weights = transmittance * alpha
pixel_color = (weights[:, None] * color).sum(dim=0)

print("ray samples:", ray_samples.shape)
print("NeRF weights:", weights)
print("rendered color:", pixel_color)


## 3. 3D Gaussian covariance from rotation and scale

3D Gaussian Splatting의 primitive는 point가 아니라 anisotropic Gaussian이다. covariance는 rotation `R`과 axis scale `s`에서 `Sigma_3D = R diag(s^2) R^T`로 만든다.


In [ ]:
scale = torch.tensor(
    [0.18, 0.08, 0.25],
    device=device,
)

angle = torch.tensor(0.4, device=device)
rotation = torch.stack(
    [
        torch.stack([angle.cos(), -angle.sin(), torch.tensor(0.0, device=device)]),
        torch.stack([angle.sin(), angle.cos(), torch.tensor(0.0, device=device)]),
        torch.tensor([0.0, 0.0, 1.0], device=device),
    ]
)

scale_covariance = torch.diag(scale.square())
covariance_3d = rotation @ scale_covariance @ rotation.T

print("3D covariance:\n", covariance_3d)


## 4. Project the 3D covariance to screen space

Gaussian center만 projection하면 splat의 크기와 방향을 알 수 없다. perspective map의 Jacobian `J`를 Gaussian mean에서 계산하고 camera-space covariance에 적용해 `Sigma_2D = J Sigma_camera J^T`를 만든다. 이것이 screen-space ellipse의 핵심이다.


In [ ]:
gaussian_mean_world = torch.tensor(
    [0.2, 0.1, 2.0],
    device=device,
)
gaussian_mean_camera = R_camera @ gaussian_mean_world + t_camera

x, y, z = gaussian_mean_camera

projection_jacobian = torch.stack(
    [
        torch.stack([
            torch.tensor(fx, device=device) / z,
            torch.tensor(0.0, device=device),
            -torch.tensor(fx, device=device) * x / z.square(),
        ]),
        torch.stack([
            torch.tensor(0.0, device=device),
            torch.tensor(fy, device=device) / z,
            -torch.tensor(fy, device=device) * y / z.square(),
        ]),
    ]
)

covariance_camera = (
    R_camera @ covariance_3d @ R_camera.T
)
covariance_2d = (
    projection_jacobian
    @ covariance_camera
    @ projection_jacobian.T
)

# Small diagonal term prevents numerical degeneracy.
covariance_2d = covariance_2d + 1e-4 * torch.eye(2, device=device)

mean_pixel = torch.stack(
    [
        fx * x / z + cx,
        fy * y / z + cy,
    ]
)

print("screen mean:", mean_pixel)
print("screen covariance:\n", covariance_2d)


## 5. Screen-space Gaussian weight becomes per-pixel alpha

pixel offset `d=p-mu_2D`에 대해 Gaussian exponent `exp(-1/2 d^T Sigma_2D^{-1} d)`를 계산하고 learned opacity를 곱하면 그 Gaussian이 해당 pixel에 기여하는 alpha가 된다.


In [ ]:
pixel = mean_pixel + torch.tensor(
    [1.0, -0.5],
    device=device,
)
offset = pixel - mean_pixel
inverse_covariance = torch.linalg.inv(covariance_2d)
mahalanobis = offset @ inverse_covariance @ offset
gaussian_weight = torch.exp(-0.5 * mahalanobis)

opacity = torch.tensor(0.7, device=device)
pixel_alpha = opacity * gaussian_weight

print("Mahalanobis distance²:", mahalanobis.item())
print("Gaussian weight:", gaussian_weight.item())
print("pixel alpha:", pixel_alpha.item())


## 6. Depth-ordered front-to-back Gaussian compositing

실제 splatting에서는 같은 tile/pixel에 영향을 주는 Gaussian들을 depth order로 정렬하고, 각 Gaussian의 screen-space alpha를 front-to-back으로 compositing한다.


In [ ]:
depth = torch.tensor([1.5, 2.0, 3.0], device=device)
alpha_values = torch.tensor([0.35, 0.5, 0.25], device=device)
colors = torch.tensor(
    [
        [1.0, 0.1, 0.1],
        [0.1, 1.0, 0.1],
        [0.1, 0.1, 1.0],
    ],
    device=device,
)

order = depth.argsort()
alpha_sorted = alpha_values[order]
color_sorted = colors[order]

survival = torch.cat(
    [
        torch.ones(1, device=device),
        1 - alpha_sorted + 1e-8,
    ]
)
transmittance = torch.cumprod(survival, dim=0)[:-1]
splat_weights = transmittance * alpha_sorted
composited_color = (
    splat_weights[:, None] * color_sorted
).sum(dim=0)

print("splat weights:", splat_weights)
print("composited color:", composited_color)


## References and provenance

**NeRF** — Mildenhall et al. ray sampling, density-to-alpha, transmittance, weighted RGB integration을 반영했다.

**3D Gaussian Splatting** — Kerbl et al. anisotropic 3D covariance, perspective Jacobian을 통한 screen-space covariance, Gaussian pixel footprint, opacity, depth-ordered alpha compositing의 핵심 계산 사슬을 반영했다. production rasterizer의 tile binning/sorting CUDA kernel은 이 tiny notebook의 범위 밖이다.
